## 1. Load Data

In [9]:
import sys
import os
sys.path.append(os.path.abspath('../src'))

import pandas as pd
from utils import get_path, load_csv, save_csv, save_pickle

train_df = load_csv(get_path("data", "processed", "train.csv"))
val_df = load_csv(get_path("data", "processed", "val.csv"))
test_df = load_csv(get_path("data", "processed", "test.csv"))

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)
train_df.head()

Train: (10564, 3)
Val: (1554, 3)
Test: (3041, 3)


,review,aspect,sentiment
0,ảnh chụp từ hôm qua đi chơi với gia đình và 1 ...,5,NaN
1,ảnh chụp từ hôm qua đi chơi với gia đình và 1 ...,6,NaN
2,hương vị thơm ngon ăn cay cay rất thích nêm nế...,0,NaN
3,hương vị thơm ngon ăn cay cay rất thích nêm nế...,4,NaN
4,hương vị thơm ngon ăn cay cay rất thích nêm nế...,5,NaN


## 2. Text Cleaning

In [10]:
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'^_+', '', text)
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

train_df['review'] = train_df['review'].apply(clean_text)
val_df['review']   = val_df['review'].apply(clean_text)
test_df['review']  = test_df['review'].apply(clean_text)

train_df['review'].sample(3).values

<ArrowStringArray>
['mình bị ghiền món này rồi cơm trộn hàn quốc siêu siêu ngon 1 tô bự lắm có trứng thịt bò kimchi giá dưa leo cà rốt rau gì đó với rong biển xong rồi mình cho thêm sốt hàn quốc rồi trộn đều lên hết món này đơn giản mà ngon dễ sợ luôn á quán này teen teen bước vô thấy cũng hao hao với hanuri treo hình mấy nhóm nhạc kpop nhưng được cái thoải mái dễ chịu hơn bên đó',
                                      'nhỏ đến lớn ghé đây mấy lần với gia đình đều chọn ăn món này mê lắm cọng bánh canh dầy chứ k mỏng lỏng lẻo như bình thường 1 tô đầy thịt nước lèo thì thơm còn gì chê nữa có thể chọn bánh canh không giò hay nạc tuỳ ý như mình thì chỉ thích nạc ăn vài cái cuốn r gọi 1 tô bánh canh ntn ra ăn xong chỉ muốn lăn chứ k đi nổi nữa',
                                                                                                                                                                                 'bán càng ngày càng điêu xiên thịt cứ teo dần đều đã thế mấy cô bán hàng

## 3. Encode Labels

In [11]:
from utils import encode_sentiment, encode_aspect

train_df, sentiment_map = encode_sentiment(train_df)
val_df, _ = encode_sentiment(val_df)
test_df, _ = encode_sentiment(test_df)

train_df, aspect_map = encode_aspect(train_df)
val_df, _ = encode_aspect(val_df)
test_df, _ = encode_aspect(test_df)

print("Sentiment mapping:", sentiment_map)
print("Aspect mapping:", aspect_map)

Sentiment mapping: {'positive': 0, 'neutral': 1, 'negative': 2}
Aspect mapping: {0: 0, 1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6, 7: 7, 8: 8, 9: 9, 10: 10, 11: 11}


In [12]:
save_pickle(sentiment_map, get_path("models", "artifacts", "sentiment_map.pkl"))
save_pickle(aspect_map, get_path("models", "artifacts", "aspect_map.pkl"))

Saved to c:\Users\thevi\Documents\Code\viet-restaurant-absa\models\artifacts\sentiment_map.pkl
Saved to c:\Users\thevi\Documents\Code\viet-restaurant-absa\models\artifacts\aspect_map.pkl


## 4. Tokenize with PhoBERT

In [13]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base")

MAX_LENGTH = 128

In [14]:
def tokenize(df, max_length=MAX_LENGTH):
    return tokenizer(
        df['review'].tolist(),
        max_length=max_length,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )

train_enc = tokenize(train_df)
val_enc = tokenize(val_df)
test_enc = tokenize(test_df)

print("input_ids shape :", train_enc['input_ids'].shape)
print("attention_mask shape:", train_enc['attention_mask'].shape)

input_ids shape : torch.Size([10564, 128])
attention_mask shape: torch.Size([10564, 128])


## 5. Verify

In [15]:
idx = 0
print("Original:", train_df['review'].iloc[idx])
print("Decoded:", tokenizer.decode(train_enc['input_ids'][idx], skip_special_tokens=True))
print("Sentiment:", train_df['sentiment'].iloc[idx])
print("Aspect:", train_df['aspect'].iloc[idx])

Original: ảnh chụp từ hôm qua đi chơi với gia đình và 1 nhà họ hàng đang sống tại sài gòn _ hôm qua đi ăn trưa muộn ai cũng đói hết nên lúc có đồ ăn là nhào vô ăn liền bởi vậy mới quên chụp các phần gọi thêm với nước mắm chỉ chụp món chính thôi _ đói quá nên không biết đánh giá đồ ăn kiểu gì luôn _ chọn cái này vì thấy nó lạ với tui
Decoded: ảnh chụp từ hôm qua đi chơi với gia đình và 1 nhà họ hàng đang sống tại sài gòn _ hôm qua đi ăn trưa muộn ai cũng đói hết nên lúc có đồ ăn là nhào vô ăn liền bởi vậy mới quên chụp các phần gọi thêm với nước mắm chỉ chụp món chính thôi _ đói quá nên không biết đánh giá đồ ăn kiểu gì luôn _ chọn cái này vì thấy nó lạ với tui
Sentiment: nan
Aspect: 5


## 6. Save Encoded Data

In [ ]:
import torch

torch.save(train_enc, get_path("data", "processed", "train_enc.pt"))
torch.save(val_enc, get_path("data", "processed", "val_enc.pt"))
torch.save(test_enc, get_path("data", "processed", "test_enc.pt"))

save_csv(train_df, get_path("data", "processed", "train_encoded.csv"))
save_csv(val_df,   get_path("data", "processed", "val_encoded.csv"))
save_csv(test_df,  get_path("data", "processed", "test_encoded.csv"))

Saved to c:\Users\thevi\Documents\Code\viet-restaurant-absa\data\processed\train.csv
Saved to c:\Users\thevi\Documents\Code\viet-restaurant-absa\data\processed\val.csv
Saved to c:\Users\thevi\Documents\Code\viet-restaurant-absa\data\processed\test.csv
